In [141]:
import pandas as pd
import numpy as np

df = pd.read_csv("atp_matches_git.csv")
df_qual = pd.read_csv("atp_qual.csv")

C:\Users\rohan\AppData\Local\Temp\ipykernel_24676\3679052332.py:4: DtypeWarning: Columns (7,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("atp_matches_git.csv")


In [142]:
df = df[df["tourney_id"].ne("tourney_id")].copy()

# Strip whitespace in all object columns
obj_cols = df.select_dtypes(include="object").columns
df[obj_cols] = df[obj_cols].apply(lambda s: s.str.strip())

df_qual = df_qual[df_qual["tourney_id"].ne("tourney_id")].copy()

# Strip whitespace in all object columns
obj_cols = df_qual.select_dtypes(include="object").columns
df_qual[obj_cols] = df_qual[obj_cols].apply(lambda s: s.str.strip())

In [143]:
# Parse tournament date (YYYYMMDD)

# Columns that should be numeric (this dataset has many)
numeric_cols = [
    "draw_size", "match_num", "best_of", "minutes",
    "winner_id", "winner_seed", "winner_ht", "winner_age",
    "loser_id", "loser_seed", "loser_ht", "loser_age",
    "winner_rank", "winner_rank_points", "loser_rank", "loser_rank_points",
    # post-match stats (keep numeric even if you later drop for leakage)
    "w_ace","w_df","w_svpt","w_1stIn","w_1stWon","w_2ndWon","w_SvGms","w_bpSaved","w_bpFaced",
    "l_ace","l_df","l_svpt","l_1stIn","l_1stWon","l_2ndWon","l_SvGms","l_bpSaved","l_bpFaced", "match_date",
]

for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
        
        
df["surface"] = df["surface"].replace({"Carpet": "Hard"})


In [144]:
df_qual = df_qual.drop('match_date', axis=1)
df_qual = df_qual.rename(columns={'match_date_yyyymmdd': 'match_date'})
for c in numeric_cols:
    if c in df_qual.columns:
        df_qual[c] = pd.to_numeric(df_qual[c], errors="coerce")
        
        
df_qual["surface"] = df_qual["surface"].replace({"Carpet": "Hard"})
df_qual["score"] = df_qual["score"].fillna("ddd")
df_qual["score"].isna().sum()

np.int64(0)

In [145]:
df = pd.concat([df, df_qual], ignore_index=True)
df = df.sort_values("tourney_date", kind="mergesort").reset_index(drop=True)
df["score"].isna().sum()
df["score"].isna().sum()
print(df["score"].isna().sum())
print(df["tourney_date"])

0
0        20170102
1        20170102
2        20170102
3        20170102
4        20170102
           ...   
34539    20251222
34540    20251222
34541    20251222
34542    20251222
34543    20251222
Name: tourney_date, Length: 34544, dtype: int64


In [146]:
for col in ["winner_age", "loser_age", 'winner_ht', 'loser_ht']:
    df[col] = df.groupby('loser_name')[col].ffill().bfill()
    
for col in ["winner_hand", "loser_hand"]:
    mode = df[col].mode(dropna=True)
    fill_val = mode.iloc[0] if not mode.empty else "R"
    df[col] = df[col].fillna(fill_val)

# -------------------------
# MINUTES (median by best_of)
# -------------------------
post_match_stats = [
    "ace", "df", "svpt", "1stIn", "1stWon",
    "2ndWon", "bpSaved", "bpFaced", "SvGms"
]

for stat in post_match_stats:
    for prefix in ["w_", "l_"]:
        col = prefix + stat
        if col in df.columns:
            df[col] = (
                df.groupby("best_of")[col]
                  .transform(lambda x: x.fillna(x.median()))
            )
            df[col] = df[col].fillna(df[col].median())

df["minutes"] = ( df.groupby("best_of")["minutes"] .transform(lambda x: x.fillna(x.median())) ) 
df["surface"] = df["surface"].fillna("Hard")
df['loser_rank_points'] = df.groupby('loser_name')['loser_rank_points'].ffill().bfill()
df['winner_rank_points'] = df.groupby('winner_name')['winner_rank_points'].ffill().bfill()


In [147]:
# No missing values left
missing_summary = (
    df.isna()
      .mean()
      .sort_values(ascending=False)
      .to_frame("missing_frac")
)

missing_summary

,missing_frac
winner_entry,0.872829
loser_entry,0.790875
loser_seed,0.725712
winner_id,0.559518
loser_id,0.559431
winner_seed,0.547736
loser_rank,0.014387
match_num,0.014156
winner_rank,0.003705
draw_size,0.000463


In [ ]:
def test_cleaned_df(df):
    """Sanity-check the cleaned dataframe before export."""
    required_cols = [
        "tourney_id", "tourney_date", "surface", "winner_name", "loser_name",
        "best_of", "score", "winner_age", "loser_age", "winner_ht", "loser_ht",
    ]
    missing_cols = [c for c in required_cols if c not in df.columns]
    assert not missing_cols, f"Missing expected columns: {missing_cols}"

    # Key columns should have no nulls after cleaning
    for col in required_cols:
        n_missing = df[col].isna().sum()
        assert n_missing == 0, f"{col} has {n_missing} missing values"

    assert set(df["surface"].unique()) <= {"Hard", "Clay", "Grass"}, "Unexpected surface value"
    assert (df["best_of"].isin([3, 5])).all(), "best_of contains values other than 3 or 5"
    assert len(df) > 0, "Cleaned dataframe is empty"

    print(f"All checks passed on {len(df)} rows.")


test_cleaned_df(df)


In [148]:
df.to_csv('atp_matches_data_cleaned.csv', index=False, header=True)